# Direct Preference Optimization (DPO) on Qwen3 with MaxText

This notebook demonstrates the end-to-end workflow of fine-tuning a model using Direct Preference Optimization (DPO) with MaxText.

DPO is a stable and computationally efficient method for aligning large language models to human preferences, bypassing the need for fitting a separate reward model or using complex reinforcement learning loops (like PPO).

### Workflow Overview:
1.  **Setup and Checkpoint Conversion**: Download a pre-trained SFT model (Qwen3-0.6B-Instruct) and convert it to MaxText format.
2.  **Baseline Evaluation**: Evaluate the SFT model's instruction-following capabilities on the `IFEval` benchmark before DPO training.
3.  **DPO Training**: Fine-tune the model on a preference dataset (`distilabel-intel-orca-dpo-pairs`) using the Tunix DPOTrainer integrated with MaxText.
4.  **Post-DPO Evaluation**: Evaluate the aligned model on `IFEval` to verify behavioral improvement.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/dpo_qwen3_demo.ipynb)

## Prerequisites

Before running this notebook, ensure you have access to a TPU (v4 or newer is recommended). 
If you are running in Google Colab, make sure to select a TPU v2-8 or TPU v4-8 runtime.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
  print("Running the notebook on local/remote VM")
  IN_COLAB = False

## Installation: MaxText and Post-training Dependencies

We need to install MaxText and additional dependencies for post-training (Tunix) and evaluation (lm-evaluation-harness).

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv

    # Install MaxText and post-training dependencies
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

# Install evaluation dependencies
try:
    import lm_eval
    import nest_asyncio
    import langdetect
except ImportError:
    if IN_COLAB:
        !uv pip install "lm_eval[api]" nest_asyncio langdetect
    else:
        %pip install "lm_eval[api]" nest_asyncio langdetect

## Environment Setup & Imports

Initialize JAX distributed system (if running on multi-host) and import necessary libraries.

In [ ]:
import jax
import os
import sys
import subprocess
import transformers
import logging
from etils import epath

# Configure logging to see evaluation progress
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

# Add src to path if not running from root
sys.path.append(os.path.abspath("src"))

from maxtext.configs import pyconfig
from maxtext.trainers.post_train.dpo import train_dpo
from maxtext.utils.globals import MAXTEXT_PKG_DIR

if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Ensure you have logged in via `huggingface-cli login` or set HF_TOKEN env var.")

## Configurations

Define the model, dataset, and paths for training and evaluation.
We will use Qwen3-0.6B-Instruct as our SFT baseline, and align it using DPO on the `distilabel-intel-orca-dpo-pairs` dataset.

In [ ]:
MODEL_NAME = "qwen3-0.6b"
TOKENIZER_PATH = "Qwen/Qwen3-0.6B"
HF_PATH = "Qwen/Qwen3-0.6B"

# Local directories
BASE_DIR = "/tmp/dpo_demo"
MODEL_CHECKPOINT_PATH = f"{BASE_DIR}/sft_baseline_checkpoint"
DPO_OUTPUT_DIR = f"{BASE_DIR}/dpo_output"
RUN_NAME = "dpo_qwen3_demo"

# Create directories
os.makedirs(BASE_DIR, exist_ok=True)

## Download and Convert SFT Checkpoint

We download the pre-trained SFT model from Hugging Face and convert it to MaxText format.
This converted checkpoint will serve as the starting point (and reference model) for DPO.

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    # Install torch for the conversion script
    print("Installing torch for conversion...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            "src/maxtext/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            f"--hf_model_path={HF_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )

    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Checkpoint converted to {CONVERTED_SFT_CKPT}")
else:
    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Model checkpoint already exists at {CONVERTED_SFT_CKPT}")

## Step 1: Evaluation before DPO (SFT Baseline)

We evaluate the converted SFT checkpoint on the `IFEval` task to establish a baseline.
We use the unified evaluation runner with `--runner lm_eval`.

In [ ]:
from maxtext.eval.runner.harness_runner import run_harness

# Configure evaluation
eval_cfg_pre = {
    "checkpoint_path": CONVERTED_SFT_CKPT,
    "model_name": MODEL_NAME,
    "hf_path": HF_PATH,
    "tasks": ["ifeval"],
    "num_samples": 20, # Keep it fast for demo
    "base_output_directory": BASE_DIR,
    "run_name": "sft_baseline",
    "results_path": f"{BASE_DIR}/sft_baseline/eval_results",
    "max_model_len": 1024,
    "tensor_parallel_size": 1,
    "hbm_memory_utilization": 0.85,
}

# Run evaluation
print("Running Pre-DPO Evaluation on IFEval...")
run_results_pre = run_harness(eval_cfg_pre)
print("Pre-DPO Evaluation Completed.")
print(f"Results saved to: {run_results_pre.get('results_file')}")
print(f"Scores: {run_results_pre.get('scores')}")

## Step 2: Direct Preference Optimization (DPO) Training

Now we run DPO training to align the model. We use the `train_dpo` script programmatically.
We configure it to run for a small number of steps for demonstration purposes.

In [ ]:
# Initialize configuration for DPO training
dpo_config = pyconfig.initialize_pydantic([
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/dpo.yml",
    f"load_parameters_path={CONVERTED_SFT_CKPT}",
    f"model_name={MODEL_NAME}",
    f"base_output_directory={DPO_OUTPUT_DIR}",
    f"run_name={RUN_NAME}",
    f"tokenizer_path={TOKENIZER_PATH}",
    "dataset_type=hf",
    "hf_path=argilla/distilabel-intel-orca-dpo-pairs",
    "train_split=train",
    "hf_eval_split=train",
    "train_data_columns=['input', 'chosen', 'rejected']",
    "per_device_batch_size=1",
    "max_target_length=1024",
    "steps=50",  # Running for 50 steps for the demo
    "eval_interval=50",
    "learning_rate=5e-7",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "scan_layers=True",
    "enable_checkpointing=True",
    "async_checkpointing=False",
])

# Setup trainer state
print("Setting up DPO trainer state...")
trainer, mesh = train_dpo.setup_trainer_state(dpo_config)

# Run DPO training
print("Starting DPO training...")
trainer = train_dpo.train_model(dpo_config, trainer, mesh)
print("DPO training completed!")

# Path to the trained DPO checkpoint
DPO_CHECKPOINT = os.path.join(DPO_OUTPUT_DIR, RUN_NAME, "checkpoints", "50", "items")
print(f"DPO checkpoint saved at: {DPO_CHECKPOINT}")

# Clean up DPO trainer state to free JAX memory references
del trainer
del mesh
import gc
gc.collect()

## Step 3: Evaluation after DPO

We evaluate the DPO-aligned model on `IFEval` to see the improvement in instruction-following performance.

In [ ]:
# Configure evaluation for DPO model
eval_cfg_post = {
    "checkpoint_path": DPO_CHECKPOINT,
    "model_name": MODEL_NAME,
    "hf_path": HF_PATH,
    "tasks": ["ifeval"],
    "num_samples": 20, # Keep it fast for demo
    "base_output_directory": BASE_DIR,
    "run_name": "dpo_aligned",
    "results_path": f"{BASE_DIR}/dpo_aligned/eval_results",
    "max_model_len": 1024,
    "tensor_parallel_size": 1,
    "hbm_memory_utilization": 0.85,
}

# Run evaluation
print("Running Post-DPO Evaluation on IFEval...")
run_results_post = run_harness(eval_cfg_post)
print("Post-DPO Evaluation Completed.")
print(f"Results saved to: {run_results_post.get('results_file')}")
print(f"Scores: {run_results_post.get('scores')}")

## Summary

In this notebook, you have:
1. Converted a Hugging Face SFT model to MaxText format.
2. Evaluated the baseline SFT model on the IFEval benchmark.
3. Aligned the model using Direct Preference Optimization (DPO) with MaxText and Tunix.
4. Evaluated the DPO-aligned model to verify improvement.

For more details on DPO configurations and scaling, refer to the [MaxText Post-Training Documentation](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/trainers/post_train/README.md).